In [ ]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"


import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,count
import requests

spark= SparkSession \
       .builder \
       .appName("Datos desde un CSV") \
       .getOrCreate()

spark

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cli.github.com/packages stable InRelease [4,685 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,182 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,965 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Get:14 http://archi

In [ ]:
import requests
import pandas as pd
from google.colab import data_table  # Para usar DataTable en Colab
import matplotlib.pyplot as plt


# Descargar el archivo CSV desde la URL -> ETL
path = "https://raw.githubusercontent.com/Data-Market/productos-de-supermercados/refs/heads/main/productos-de-supermercado-sample.csv"
req = requests.get(path) # Extrae
url_content = req.content # CSV Transformación
name_file = "ingesta_git_supermercados.csv" # Gobierno
csv_file = open(name_file, "wb")
csv_file.write(url_content) # Similar a la transmisión

#Mover archivo
import os
import shutil

os.makedirs('/content/staging/', exist_ok=True)
shutil.move('/content/ingesta_git_supermercados.csv','/content/staging/ingesta_git_supermercados.csv')


'/content/staging/ingesta_git_supermercados.csv'

In [ ]:
spark.sparkContext.defaultParallelism

2

In [ ]:
# Ingesta Master
df = spark.read.csv('/content/staging/' + name_file, header=True, inferSchema=True)
df.drop("reference_price").where(df.supermarket == "carrefour-es").coalesce(1).write\
  .partitionBy("supermarket").mode("append")\
  .parquet('/content/master/ingesta_git_supermercados')

  #

In [ ]:
#Ejemplos de lectura
spark.read\
  .option("basePath","/content/master/ingesta_git_supermercados/")\
  .parquet("/content/master/ingesta_git_supermercados/supermarket=dia-es/").printSchema()

root
 |-- category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- price: string (nullable = true)
 |-- reference_price: string (nullable = true)
 |-- reference_unit: string (nullable = true)
 |-- insert_date: timestamp (nullable = true)
 |-- supermarket: string (nullable = true)



In [ ]:
#Ejemplos de lectura
spark.read\
  .option("basePath","/content/master/ingesta_git_supermercados/")\
  .parquet("/content/master/ingesta_git_supermercados/supermarket=carrefour-es/").printSchema()

root
 |-- category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- price: string (nullable = true)
 |-- reference_unit: string (nullable = true)
 |-- insert_date: timestamp (nullable = true)
 |-- supermarket: string (nullable = true)



In [ ]:
spark.read.parquet("/content/master/ingesta_git_supermercados/").printSchema()

root
 |-- category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- price: string (nullable = true)
 |-- reference_unit: string (nullable = true)
 |-- insert_date: timestamp (nullable = true)
 |-- supermarket: string (nullable = true)



In [ ]:
#Ejemplos de lectura
df_dia = spark.read\
  .option("basePath","/content/master/ingesta_git_supermercados/")\
  .parquet("/content/master/ingesta_git_supermercados/supermarket=dia-es/")

#Ejemplos de lectura
df_carrefour=spark.read\
  .option("basePath","/content/master/ingesta_git_supermercados/")\
  .parquet("/content/master/ingesta_git_supermercados/supermarket=carrefour-es/")



In [ ]:
 from pyspark.sql.functions import col,lit

 df_dia.union(df_carrefour.select(
     col("category"),
     col("name"),
     col("description"),
     col("price"),
     lit("NA").alias("reference_price"),
     col("reference_unit"),
     col("insert_date"),
     col("supermarket")
 )).show()

+--------------------+--------------------+-----------+-----+---------------+--------------+-------------------+-----------+
|            category|                name|description|price|reference_price|reference_unit|        insert_date|supermarket|
+--------------------+--------------------+-----------+-----+---------------+--------------+-------------------+-----------+
|perfumeria_e_higi...|PHARMALINE gel de...|       NULL| 5.25|            7.0|          €/l.|2021-01-05 00:00:00|     dia-es|
|perfumeria_e_higi...|ORAL B pasta dent...|       NULL| 3.99|           53.2|          €/l.|2020-12-11 12:00:00|     dia-es|
|alimentacion_cons...|ISABEL mejillones...|       NULL| 1.85|          26.81|         €/Kg.|2021-01-07 00:00:00|     dia-es|
|perfumeria_e_higi...|DAEN cera depilat...|       NULL| 4.59|          22.95|         €/Kg.|2020-10-18 12:00:00|     dia-es|
|perfumeria_e_higi...|DOVE jabón de man...|       NULL| 1.05|           10.5|         €/Kg.|2020-12-21 00:00:00|     dia-es|


In [ ]:
df = spark.read.csv('/content/staging/' + name_file, header=True, inferSchema=True)


df_pandas = df.toPandas()

# Usar DataTable para mostrar el DataFrame como si fuera una tabla de Excel
data_table.DataTable(df_pandas, include_index=False, num_rows_per_page=10)

,supermarket,category,name,description,price,reference_price,reference_unit,insert_date
0,mercadona-es,postres_y_yogures_yogures_liquidos,Bebida láctea sin lactosa de fresa Hacendado,Pack-4,1.4,2.19,kg,2020-12-23 00:00:00
1,mercadona-es,bodega_licores,Ginebra 15 botanicals Blumara,Botella,10.75,15.36,L,2020-11-06 12:00:00
2,mercadona-es,charcuteria_y_quesos_queso_untable_y_fresco,Queso fresco batido desnatado Hacendado 0% mat...,Tarrina,1.09,2.18,kg,2020-09-23 16:06:00
3,carrefour-es,la_despensa_yogures_y_postres_yogures_desnatados,Yogur bífidus desnatado con lima y limón Danon...,Pack 4x120 G.,2.08,4.33,kg,2020-10-10 00:00:00
4,carrefour-es,la_despensa_helados_bombon,Helado After Dinner Magnum sin gluten 10 ud.,10 Ud.,4.6,0.46,ud,2020-10-15 00:00:00
...,...,...,...,...,...,...,...,...
150002,mercadona-es,bodega_cerveza,Cerveza Clásica Steinburg,Botella,0.7,0.7,L,2020-12-11 00:00:00
150003,mercadona-es,mascotas_gato,Gelatina gato Felix selección de sabores,Paquete,1.49,3.73,kg,2020-12-16 04:00:00
150004,carrefour-es,mascotas_gatos_pienso_para_gatos,Ultima Pienso para Gato Esterilizado Sabor pol...,3 Kg.,14.25,4.75,kg,2020-10-31 00:00:00
150005,dia-es,bebidas_refrescos_naranja,DIA refresco de naranja zero botella 2 lt,None,0.79,0.4,€/l.,2020-08-01 16:00:00


In [ ]:
df.printSchema()

root
 |-- supermarket: string (nullable = true)
 |-- category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- price: string (nullable = true)
 |-- reference_price: string (nullable = true)
 |-- reference_unit: string (nullable = true)
 |-- insert_date: timestamp (nullable = true)



In [ ]:
df.count()

150007

In [ ]:
from pyspark.sql.functions import col

df.select("supermarket").where(col("supermarket").isNull()).count()

0

In [ ]:
df.select("supermarket").distinct().show()

+--------------------+
|         supermarket|
+--------------------+
|Licor del Polo 50...|
|        carrefour-es|
|                 g."|
|              dia-es|
|        mercadona-es|
+--------------------+



In [ ]:
from pyspark.sql.functions import count
df.select("supermarket").groupBy("supermarket").agg(count("*")).show()

+--------------------+--------+
|         supermarket|count(1)|
+--------------------+--------+
|Licor del Polo 50...|       3|
|        carrefour-es|   88444|
|                 g."|       4|
|              dia-es|   31674|
|        mercadona-es|   29882|
+--------------------+--------+



In [ ]:
# df.select("*").show() = df.show()
# select * from df

+------------+--------------------+--------------------+-------------+-----+---------------+--------------+-------------------+
| supermarket|            category|                name|  description|price|reference_price|reference_unit|        insert_date|
+------------+--------------------+--------------------+-------------+-----+---------------+--------------+-------------------+
|mercadona-es|postres_y_yogures...|Bebida láctea sin...|       Pack-4|  1.4|           2.19|            kg|2020-12-23 00:00:00|
|mercadona-es|      bodega_licores|Ginebra 15 botani...|      Botella|10.75|          15.36|             L|2020-11-06 12:00:00|
|mercadona-es|charcuteria_y_que...|Queso fresco bati...|      Tarrina| 1.09|           2.18|            kg|2020-09-23 16:06:00|
|carrefour-es|la_despensa_yogur...|Yogur bífidus des...|Pack 4x120 G.| 2.08|           4.33|            kg|2020-10-10 00:00:00|
|carrefour-es|la_despensa_helad...|Helado After Dinn...|       10 Ud.|  4.6|           0.46|            

In [ ]:
df.where(~col("supermarket").isin("carrefour-es","dia-es","mercadona-es")).show(truncate=False)
# Quedarme con todos los registros que esten dentro de mi lista
# ~ Quedarme con todos los registros que NO estan dentro de mi lista

+-----------------------+--------+----+-----------+-----+-------------------+--------------+-----------+
|supermarket            |category|name|description|price|reference_price    |reference_unit|insert_date|
+-----------------------+--------+----+-----------+-----+-------------------+--------------+-----------+
| g."                   |NULL    |1.99|4.15       |kg   |2021-01-06 00:00:00|NULL          |NULL       |
|Licor del Polo 500 ml."|NULL    |4.35|0.87       |100ml|2021-01-07 00:00:00|NULL          |NULL       |
| g."                   |NULL    |1.99|4.15       |kg   |2020-12-24 00:00:00|NULL          |NULL       |
|Licor del Polo 500 ml."|NULL    |4.35|0.87       |100ml|2021-01-16 00:00:00|NULL          |NULL       |
| g."                   |NULL    |1.95|4.06       |kg   |2021-01-27 12:00:00|NULL          |NULL       |
| g."                   |NULL    |1.95|4.06       |kg   |2021-01-28 00:00:00|NULL          |NULL       |
|Licor del Polo 500 ml."|NULL    |4.35|0.87       |100m

In [ ]:
df.where(col("supermarket").isin("carrefour-es","dia-es","mercadona-es")).show() # muestra 20 registros
# df.where(col("supermarket").isin("carrefour-es","dia-es","mercadona-es")).show(1) # Muestra 1 registro

+------------+--------------------+--------------------+-------------+-----+---------------+--------------+-------------------+
| supermarket|            category|                name|  description|price|reference_price|reference_unit|        insert_date|
+------------+--------------------+--------------------+-------------+-----+---------------+--------------+-------------------+
|mercadona-es|postres_y_yogures...|Bebida láctea sin...|       Pack-4|  1.4|           2.19|            kg|2020-12-23 00:00:00|
|mercadona-es|      bodega_licores|Ginebra 15 botani...|      Botella|10.75|          15.36|             L|2020-11-06 12:00:00|
|mercadona-es|charcuteria_y_que...|Queso fresco bati...|      Tarrina| 1.09|           2.18|            kg|2020-09-23 16:06:00|
|carrefour-es|la_despensa_yogur...|Yogur bífidus des...|Pack 4x120 G.| 2.08|           4.33|            kg|2020-10-10 00:00:00|
|carrefour-es|la_despensa_helad...|Helado After Dinn...|       10 Ud.|  4.6|           0.46|            

In [ ]:
# Tipos de dato
from pyspark.sql.functions import col

df_clean = df.where(col("supermarket").isin("carrefour-es","dia-es","mercadona-es"))
df_clean.printSchema()
df_clean.withColumn("price", col("price").cast("double") ).printSchema()

# String (Hola) -> cast(int) => No se puede
# String (123) -> cast (int) => (123)
# String (123.4) -> cast(int) => No se puede
# String (123.4) -> cast(double) => (123.4) -> cast(int) => (123)

root
 |-- supermarket: string (nullable = true)
 |-- category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- price: string (nullable = true)
 |-- reference_price: string (nullable = true)
 |-- reference_unit: string (nullable = true)
 |-- insert_date: timestamp (nullable = true)

root
 |-- supermarket: string (nullable = true)
 |-- category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- price: double (nullable = true)
 |-- reference_price: string (nullable = true)
 |-- reference_unit: string (nullable = true)
 |-- insert_date: timestamp (nullable = true)



In [ ]:
#Esta consulta no esta optimizada
df.where(col("supermarket").isin("carrefour-es","dia-es","mercadona-es"))\
      .withColumn("price", col("price").cast("float") )\
      .withColumn("reference_price", col("reference_price").cast("float") ).printSchema()

root
 |-- supermarket: string (nullable = true)
 |-- category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- price: float (nullable = true)
 |-- reference_price: float (nullable = true)
 |-- reference_unit: string (nullable = true)
 |-- insert_date: timestamp (nullable = true)



In [ ]:
df_con_col_duplicadas = df.where(col("supermarket").isin("carrefour-es","dia-es","mercadona-es"))\
    .select(
        col("*"),
        col("price").cast("double").alias("price"),
        col("reference_price").cast("double").alias("reference_price")
    )
df_con_col_duplicadas.show()

+------------+--------------------+--------------------+-------------+-----+---------------+--------------+-------------------+-----+---------------+
| supermarket|            category|                name|  description|price|reference_price|reference_unit|        insert_date|price|reference_price|
+------------+--------------------+--------------------+-------------+-----+---------------+--------------+-------------------+-----+---------------+
|mercadona-es|postres_y_yogures...|Bebida láctea sin...|       Pack-4|  1.4|           2.19|            kg|2020-12-23 00:00:00|  1.4|           2.19|
|mercadona-es|      bodega_licores|Ginebra 15 botani...|      Botella|10.75|          15.36|             L|2020-11-06 12:00:00|10.75|          15.36|
|mercadona-es|charcuteria_y_que...|Queso fresco bati...|      Tarrina| 1.09|           2.18|            kg|2020-09-23 16:06:00| 1.09|           2.18|
|carrefour-es|la_despensa_yogur...|Yogur bífidus des...|Pack 4x120 G.| 2.08|           4.33|        

In [ ]:
# df_con_col_duplicadas.where(col("price")>0)

{"ts": "2026-08-21 03:18:16.100", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[AMBIGUOUS_REFERENCE] Reference `price` is ambiguous, could be: [`price`, `price`]. SQLSTATE: 42704", "context": {"file": "line 1 in cell [61]", "line": "", "fragment": "col", "errorClass": "AMBIGUOUS_REFERENCE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1608.filter.\n: org.apache.spark.sql.AnalysisException: [AMBIGUOUS_REFERENCE] Reference `price` is ambiguous, could be: [`price`, `price`]. SQLSTATE: 42704\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.ambiguousReferenceError(QueryCompilationErrors.scala:2163)\n\tat org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.resolve(package.scala:363)\n\tat org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.resolveChildren(LogicalPlan.scala:164)\n\tat org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpressionByPlanChildren$1(ColumnResolutionHelper.

AnalysisException: [AMBIGUOUS_REFERENCE] Reference `price` is ambiguous, could be: [`price`, `price`]. SQLSTATE: 42704

In [ ]:
df_optimizado = df.where(col("supermarket").isin("carrefour-es","dia-es","mercadona-es"))\
    .select(
        col("supermarket"),
        col("category"),
        col("name"),
        col("description"),
        col("price").cast("double").alias("price"),
        col("reference_price").cast("double").alias("reference_price"),
        col("reference_unit"),
          col("insert_date")
    )
df_optimizado.printSchema()

root
 |-- supermarket: string (nullable = true)
 |-- category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- price: double (nullable = true)
 |-- reference_price: double (nullable = true)
 |-- reference_unit: string (nullable = true)
 |-- insert_date: timestamp (nullable = true)

